# InstaNovo fine-tuning — Ecoli_EV_1

Fine-tunes InstaNovo (transformer) on the held-in 85% fraction of E. coli EV_1.
Saves the resulting checkpoint to Drive for the evaluation and prediction
notebooks to consume.

**Inputs (must exist on Drive — sync from your project before running):**
- `MyDrive/DL-Project/data_mgf_annotated/ecoli/Ecoli_EV_1.instanovo.train.mgf`
- `MyDrive/DL-Project/data_mgf_annotated/ecoli/Ecoli_EV_1.instanovo.val.mgf`
- `MyDrive/DL-Project/bin/instanovo/instanovo-v1.2.0.ckpt`
- `MyDrive/DL-Project/config/finetune/instanovo.yaml`  (the Hydra config)

**Output (written to Drive):**
- `MyDrive/DL-Project/model_finetune/instanovo/`  (fine-tuned checkpoint + logs)

**Schedule (300 steps, ~30 steps/epoch → ~10 epochs):**
- Epochs 0–1 (steps 0–59): head only (LR warmup over first 30 steps)
- Epochs 2–4 (steps 60–149): decoder added
- Epochs 5–9 (steps 150–299): everything trains

Held-out test (`Ecoli_EV_2`) is *not* loaded here — that's consumed by the
evaluate / predict notebooks downstream.


In [9]:
!nvidia-smi

Sat May  2 17:45:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Install dependencies


In [10]:
try:
  import instanovo
  !instanovo version
except ImportError:
  !pip install "instanovo[cu126]>=1.2.2" pyopenms-viz
  print('Installation complete. Restarting runtime to apply changes...')
  import os
  os.kill(os.getpid(), 9)

┏━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Package    ┃ Version ┃
┡━━━━━━━━━━━━╇━━━━━━━━━┩
│ InstaNovo  │ 1.2.2   │
│ InstaNovo+ │ 1.2.2   │
│ NumPy      │ 2.2.6   │
│ PyTorch    │ 2.8.0   │
└────────────┴─────────┘


## Sync inputs from Drive

Pulls the train/val annotated MGFs, the pretrained checkpoint, and the
Hydra config out of Drive into Colab's local filesystem.

**About the Hydra config path:** InstaNovo's CLI requires `--config-path`
to be *relative* (Hydra's `initialize()` API doesn't accept absolute paths
like `/content/config/...`). The cleanest workaround is to drop our config
into InstaNovo's own `configs/` directory — then we can pass just `-cn`
(no `-cp`) and Hydra finds it on the default search path.


In [11]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, instanovo
os.makedirs('/content/data_mgf_annotated/ecoli', exist_ok=True)
os.makedirs('/content/bin/instanovo', exist_ok=True)
os.makedirs('/content/config/finetune', exist_ok=True)
os.makedirs('/content/config/residues', exist_ok=True)
os.makedirs('/content/model_finetune/instanovo', exist_ok=True)

# Annotated train + val MGFs
!cp /content/drive/MyDrive/DL-Project/data_mgf_annotated/ecoli/Ecoli_EV_1.instanovo.train.mgf /content/data_mgf_annotated/ecoli/
!cp /content/drive/MyDrive/DL-Project/data_mgf_annotated/ecoli/Ecoli_EV_1.instanovo.val.mgf   /content/data_mgf_annotated/ecoli/

# Pretrained checkpoint from Drive — adjust SOURCE filename if yours differs.
# (Run `!ls /content/drive/MyDrive/DL-Project/bin/instanovo/` first if unsure.)
!cp /content/drive/MyDrive/DL-Project/bin/instanovo/instanovo-v1.2.0.ckpt /content/bin/instanovo/instanovo-v1.2.0.ckpt

# Hydra configs — copy into InstaNovo's package configs/ dir so Hydra finds
# them via -cn / defaults (relative-path requirement). Also keep /content/
# copies for inspection in the next cell.
package_configs = os.path.join(os.path.dirname(instanovo.__file__), 'configs')
print('Package configs dir:', package_configs)

# Main finetune config
!cp /content/drive/MyDrive/DL-Project/config/finetune/instanovo.yaml /content/config/finetune/instanovo.yaml
!cp /content/drive/MyDrive/DL-Project/config/finetune/instanovo.yaml {package_configs}/instanovo_ecoli_finetune.yaml

# Custom residues file — 30-residue vocab matching the v1.2.0 ckpt (the
# package default's 32-residue config doesn't match the ckpt's vocab and
# triggers a buggy resize path in the trainer). Same file is shared with
# the InstaNovoPlus finetune since v1.1.0 InstaNovo+ uses the same vocab.
# Referenced via `defaults: - override residues: 30aa` in our finetune YAML.
!cp /content/drive/MyDrive/DL-Project/config/residues/30aa.yaml /content/config/residues/30aa.yaml
!cp /content/drive/MyDrive/DL-Project/config/residues/30aa.yaml {package_configs}/residues/30aa.yaml

print('--- inputs ---')
!ls -lh /content/data_mgf_annotated/ecoli/
!ls -lh /content/bin/instanovo/
!ls -lh {package_configs}/instanovo_ecoli_finetune.yaml
!ls -lh {package_configs}/residues/30aa.yaml


Mounted at /content/drive
Package configs dir: /usr/local/lib/python3.12/dist-packages/instanovo/configs
--- inputs ---
total 66M
-rw------- 1 root root  27M May  2 17:45 Ecoli_EV_1.instanovo.train.mgf
-rw------- 1 root root 5.0M May  2 17:45 Ecoli_EV_1.instanovo.val.mgf
-rw------- 1 root root  34M May  2 16:54 Ecoli_EV_2.instanovo.annotated.mgf
total 362M
-rw------- 1 root root 362M May  2 17:45 instanovo-v1.2.0.ckpt
-rw------- 1 root root 2.2K May  2 17:45 /usr/local/lib/python3.12/dist-packages/instanovo/configs/instanovo_ecoli_finetune.yaml
-rw------- 1 root root 1.5K May  2 17:45 /usr/local/lib/python3.12/dist-packages/instanovo/configs/residues/30aa.yaml


## Patch the ckpt: wrap residues for trainer compatibility

InstaNovo's `load_model_state` reads `model_residues` from
`ckpt["residues"]["residues"]` (two levels nested), but the v1.2.0 ckpt
stores residues flat at `ckpt["residues"]` (one level). The nested `.get`
returns `{}`, so `model_residues != current_residues` is always True,
the trainer runs `_update_vocab` with default `resolution="delete"`, and
head/aa_embed get stripped from the loaded state — leaving them at random
init regardless of how we configure things at the YAML level.

This cell wraps the ckpt's residues dict once so the trainer's path lines
up. After patching, `model_residues` reads as the 30-residue dict and
matches our `residues: 30aa` config exactly → the residue-mismatch
branch doesn't fire → head/aa_embed load cleanly.

In [12]:
import torch
from omegaconf import DictConfig, OmegaConf

ckpt_path = '/content/bin/instanovo/instanovo-v1.2.0.ckpt'
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

residues = ckpt.get('residues')

# Normalize: in the v1.2.0 ckpt, `residues` is an OmegaConf DictConfig (not
# plain dict) holding the 30 residues flat. Convert to plain dict for
# downstream comparisons.
if isinstance(residues, DictConfig):
    residues = OmegaConf.to_container(residues, resolve=True)

if isinstance(residues, dict):
    if 'residues' in residues:
        # Already nested — just make sure inner is plain dict
        inner = residues['residues']
        if isinstance(inner, DictConfig):
            inner = OmegaConf.to_container(inner, resolve=True)
            ckpt['residues'] = {'residues': inner}
            torch.save(ckpt, ckpt_path)
            print(f'Re-saved with {len(inner)} residues nested + normalized to plain dict.')
        else:
            print(f'Ckpt already wrapped: {len(inner)} residues nested correctly.')
    else:
        # Flat — wrap it so load_model_state can find ckpt["residues"]["residues"]
        n = len(residues)
        print(f'Wrapping {n} flat residue entries in a `residues` sub-dict.')
        ckpt['residues'] = {'residues': residues}
        torch.save(ckpt, ckpt_path)
        print(f'Saved patched ckpt to {ckpt_path}')
else:
    print(f'Unexpected residues structure: type={type(residues)} value={residues!r}')
    print('Manual inspection needed before training.')

Wrapping 30 flat residue entries in a `residues` sub-dict.
Saved patched ckpt to /content/bin/instanovo/instanovo-v1.2.0.ckpt


## Verify the synced config

Quick sanity check that the config landed in Colab unchanged from your
project's `config/finetune/instanovo.yaml`. Skip if you trust the sync.


In [13]:
!cat /content/config/finetune/instanovo.yaml

defaults:
  - instanovo
  - finetune: default
  - override residues: 30aa   # 30-residue vocab matching the v1.2.0 ckpt — `override` because the `instanovo` parent already includes `residues: default`
  - _self_

# Fine-tuning trigger — points at the pretrained ckpt copied from Drive.
resume_checkpoint_path: /content/bin/instanovo/instanovo-v1.2.0.ckpt

# Fine-tuning hyperparameters (small data -> low LR, short schedule).
# ~960 train spectra / 32 batch ~= 30 steps/epoch x 10 epochs ~= 300 steps.
learning_rate: 5.0e-6
warmup_iters: 30          # 1 epoch warmup (LR ramps 0 -> 5e-6)
training_steps: 300
train_batch_size: 32
validation_interval: 30
checkpoint_interval: 30
console_logging_steps: 10
tensorboard_logging_steps: 10
model_save_folder_path: /content/model_finetune/instanovo

# Logging / runtime
use_neptune: false
compile_model: false
seed: 42

# Layer-freezing schedule (overrides finetune/default.yaml's defaults):
#   steps 0-59   (~epochs 0-1): head only         (warmup happens 

## Run fine-tuning

Hydra resolves `instanovo_ecoli_finetune.yaml` from InstaNovo's package
configs directory (where we copied it), composes it on top of the
default `instanovo` + `finetune: default` configs, and trains.

The CLI overrides on the next cell delete five InstaNovo-internal
validation entries that Hydra dict-merges in by default. See the
comment on that cell for why.

Three other configuration choices baked into the YAML, worth knowing
about if you ever modify it:

- **`dataset.lazy_loading: true`** — required. The trainer's
  `.shuffle(buffer_size=...)` call only works on `IterableDataset`,
  which lazy mode produces. With `lazy_loading: false` the dataset is
  a regular `Dataset` and `.shuffle()` rejects `buffer_size`.
- **`finetune.unfreeze_format: start_step`** (not `start_epoch`) —
  the trainer doesn't pass `steps_per_epoch` to `FinetuneScheduler`,
  so any epoch-based schedule raises a ValueError. Our step boundaries
  (0 / 60 / 150) encode the same 0 / 2 / 5-epoch schedule.
- **`dataset.residue_remapping`** — translates PEAKS-notation mods in
  our annotated MGF (`M(+15.99)`, `C(+57.02)`, etc.) to InstaNovo's
  UNIMOD bracket form (`M[UNIMOD:35]`, `C[UNIMOD:4]`, etc.) at load
  time. Same mechanism the package's default config uses for its own
  legacy mod tags.


In [14]:
# Hydra/OmegaConf merges dicts (it doesn't replace them) when composing
# defaults + our YAML. Our `dataset.valid_path.ecoli: ...` got *added* to
# the default dict's 5 InstaNovo-internal parquet entries instead of
# replacing them, so the loader then tries to read those (non-existent)
# files. Use `~key.path` to delete them at the CLI.
!instanovo transformer train -cn instanovo_ecoli_finetune \
    ~dataset.valid_path.acpt \
    ~dataset.valid_path.phospho \
    ~dataset.valid_path.pride \
    ~dataset.valid_path.massivekb \
    ~dataset.valid_path.lcfm

[05/02/26 17:45:48] INFO     Initializing InstaNovo training.                                                                                                                  
[05/02/26 17:45:50] INFO     NumExpr defaulting to 12 threads.                                                                                                                 
2026-05-02 17:45:53.146859: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 17:45:53.221127: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-05-02 17:45

## Push fine-tuned checkpoint + logs to Drive

Other notebooks (evaluate / predict) read from this Drive path. Re-running
this cell after a fresh fine-tune overwrites previous artifacts.


In [15]:
!mkdir -p /content/drive/MyDrive/DL-Project/model_finetune/instanovo
!cp -r /content/model_finetune/instanovo/. /content/drive/MyDrive/DL-Project/model_finetune/instanovo/
!ls -lh /content/drive/MyDrive/DL-Project/model_finetune/instanovo/

total 724M
drwx------ 4 root root 4.0K May  2 17:48 accelerator_state
-rw------- 1 root root 362M May  2 17:48 model_best.ckpt
-rw------- 1 root root 362M May  2 17:48 model_latest.ckpt
